# FinQA QLoRA fine-tune (Colab)

Fine-tunes **Qwen3-8B** with QLoRA to write FinQA programs, picks the best checkpoint on
the **dev** split, merges it into the base model and exports a **Q4_K_M GGUF** (the same
4-bit format as Ollama's `qwen3:8b`) for the laptop. A separate LoRA adapter is not an
option: Ollama 0.40 refuses `ADAPTER` ("LoRA adapters are no longer supported", checked on
the laptop).

The **test** split is never downloaded here. Score the result on the laptop with
`scripts/finqa_baseline.py --model qwen3-finqa --modes program`, against the baseline in
`docs/results/finqa_baseline.md` (59% `program`, 71% `direct`, within 1%).

**Before running**
* Runtime: **A100** recommended (about 1-1.5 h per epoch). L4 works but is about 3x slower.
  T4 is not supported (no bf16).
* The branch named in `BRANCH` must be pushed: the notebook clones the repo so training uses
  exactly the prompt, executor and scoring the baseline used (`auto_researcher.llm.finqa`).
* Leave `SMOKE = True` for the first run (about 30-40 minutes, mostly model download and
  the GGUF export). It trains a few steps on 64 examples and runs every later step, so
  problems show up before the GPU hours are spent. Then set `SMOKE = False`, restart the
  runtime and run all cells again.
* Set `HF_REPO` to upload the 5 GB GGUF to a private Hugging Face repo (the reliable way to
  get it to the laptop). It needs a write token: Colab secret `HF_TOKEN`, or paste it when asked.

**Method** (book ch. 3, section 3.5.5): 4-bit NF4 base, LoRA rank 16 / alpha 32 / dropout 0.05 on the
attention and MLP projections, learning rate 2e-4 with cosine decay, effective batch 16,
2 epochs. Loss is on the answer only (`Program: ...`), not the prompt. The prompt is rendered
exactly as Ollama renders it for `qwen3:8b` with thinking off (checked token-for-token on the
laptop), so training and serving see the same tokens.

In [ ]:
# ---- Configuration -----------------------------------------------------------
SMOKE = True                     # first run: tiny end-to-end check. Then False.

REPO_URL = "https://github.com/AndrewFSee/auto_researcher.git"
BRANCH = "finqa-finetune"        # must contain src/auto_researcher/llm/finqa.py
BASE_MODEL = "Qwen/Qwen3-8B"     # the model behind Ollama's qwen3:8b

EPOCHS = 2
LR = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
BATCH, GRAD_ACCUM = 1, 16        # effective batch 16
CHECKPOINTS_PER_EPOCH = 2        # each is scored on dev; the best is exported
DEV_EVAL_N = 300                 # dev questions generated per checkpoint (selection)
DEV_LOSS_N = 200                 # dev questions for the eval-loss curve
SEED = 0

HF_REPO = None                   # e.g. "your-hf-user/qwen3-8b-finqa" (created private)
USE_DRIVE = False                # keep checkpoints on Google Drive so a disconnect can resume

OUT = "/content/finqa_qlora"
if USE_DRIVE:
    from google.colab import drive   # may not be supported from the VS Code extension
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/finqa_qlora"
if SMOKE:
    OUT += "_smoke"
    HF_REPO = HF_REPO and HF_REPO + "-smoke"
GGUF_NAME = "qwen3-8b-finqa-Q4_K_M.gguf"
print("artifacts ->", OUT)

## 1. Environment

In [ ]:
import torch

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100 (or L4)"
assert torch.cuda.is_bf16_supported(), "This GPU has no bf16 (a T4?): use an A100 or L4 runtime"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
GEN_BATCH = 8 if GPU_GB > 30 else 4
print(f"{torch.cuda.get_device_name(0)}: {GPU_GB:.0f} GB, generation batch {GEN_BATCH}")

%pip install -q -U "transformers>=4.56" "peft>=0.15" "bitsandbytes>=0.45" accelerate

## 2. Code and data

Clones the repo for the shared prompt and executor, and downloads FinQA **train** and **dev** only.

In [ ]:
import hashlib, json, os, random, subprocess, sys, urllib.request
from pathlib import Path

REPO_DIR = "/content/auto_researcher"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
from auto_researcher.llm.finqa import (
    PROGRAM_PROMPT, ProgramError, answers_match, build_messages, execute_program,
    extract_program, load_finqa,
)

PROMPT_SHA = hashlib.sha256(PROGRAM_PROMPT.encode()).hexdigest()
# Same digest as tests/test_finqa.py: the prompt the baseline was run with.
assert PROMPT_SHA == "71e761a062a815845bb39e8f18b006fbc9b807f22a0c3f0d3b4c00b6ff588edb", PROMPT_SHA
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                             capture_output=True, text=True, check=True).stdout.strip()

DATA = Path("/content/finqa_data")
DATA.mkdir(exist_ok=True)
for split in ("train", "dev"):  # the test split is deliberately not downloaded
    if not (DATA / f"{split}.json").exists():
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/{split}.json",
            DATA / f"{split}.json")
train_ex = load_finqa(DATA / "train.json")
dev_ex = load_finqa(DATA / "dev.json")

rng = random.Random(SEED)
dev_pool = rng.sample(dev_ex, len(dev_ex))
if SMOKE:
    train_ex, DEV_EVAL_N, DEV_LOSS_N = train_ex[:64], 16, 16
dev_gen, dev_loss = dev_pool[:DEV_EVAL_N], dev_pool[:DEV_LOSS_N]
Path(OUT).mkdir(parents=True, exist_ok=True)
print(f"repo {REPO_COMMIT[:10]}  train {len(train_ex)}  dev gen {len(dev_gen)}  dev loss {len(dev_loss)}")

## 3. Prompt format

Ollama's `qwen3:8b` template with `think: false` appends ` /no_think` to the last user
message and opens the assistant turn with an empty think block. This was verified on the
laptop: the Hugging Face template with `enable_thinking=False` plus that suffix matches
Ollama's rendered prompt token for token (1138/1138 tokens). Without the suffix it differs.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
IM_END = "<|im_end|>"


def render_prompt(ex):
    msgs = build_messages(ex)
    msgs[-1] = {**msgs[-1], "content": msgs[-1]["content"] + " /no_think"}
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)


def render_target(ex):
    return f"Program: {ex.program}{IM_END}"


sample = render_prompt(train_ex[0])
assert sample.endswith(" /no_think<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"), sample[-200:]
print("..." + sample[-300:] + render_target(train_ex[0]))


def tokenize(ex):
    # Prompt and target are tokenized separately, as at inference (the prompt ends the
    # context and the model generates from there). Loss only on the target.
    p = tok(render_prompt(ex), add_special_tokens=False)["input_ids"]
    t = tok(render_target(ex), add_special_tokens=False)["input_ids"]
    return {"input_ids": p + t, "labels": [-100] * len(p) + t}


train_ds = [tokenize(e) for e in train_ex]
eval_ds = [tokenize(e) for e in dev_loss]
lengths = sorted(len(r["input_ids"]) for r in train_ds)
print(f"train tokens {sum(lengths)/1e6:.1f}M, median {lengths[len(lengths)//2]}, max {lengths[-1]}")

## 4. Model (4-bit base + LoRA)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16),
    dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

## 5. Dev scoring (generation + execution)

The same parsing, executor and match rules as the laptop baseline.

In [ ]:
from transformers import GenerationConfig

GEN_CONFIG = GenerationConfig(
    max_new_tokens=128, do_sample=False,
    eos_token_id=[tok.convert_tokens_to_ids(IM_END), tok.eos_token_id],
    pad_token_id=tok.pad_token_id)
MODES = ("exact", "scale", "tolerant")


@torch.no_grad()
def dev_eval(model, examples, label):
    model.eval()
    tok.padding_side = "left"
    recs = []
    for i in range(0, len(examples), GEN_BATCH):
        chunk = examples[i:i + GEN_BATCH]
        enc = tok([render_prompt(e) for e in chunk], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        out = model.generate(**enc, generation_config=GEN_CONFIG, use_cache=True)
        for ex, seq in zip(chunk, out[:, enc["input_ids"].shape[1]:]):
            text = tok.decode(seq, skip_special_tokens=True)
            program, pred, error = extract_program(text), None, None
            if program is None:
                error = "no program line"
            else:
                try:
                    pred = execute_program(program, ex.table)
                except ProgramError as exc:
                    error = f"exec: {exc}"
            recs.append({"id": ex.id, "response": text, "program": program, "pred": pred,
                         "error": error, "gold": ex.answer, "gold_program": ex.program,
                         **{m: answers_match(pred, ex.answer, m) for m in MODES}})
    tok.padding_side = "right"
    summary = {m: sum(r[m] for r in recs) / len(recs) for m in MODES}
    summary["failed"] = sum(r["error"] is not None for r in recs) / len(recs)
    summary["n"] = len(recs)
    with open(f"{OUT}/dev_{label}.jsonl", "w") as fh:
        fh.writelines(json.dumps(r) + "\n" for r in recs)
    print(f"{label:>14}: exact {summary['exact']:.1%}  scale {summary['scale']:.1%}  "
          f"within 1% {summary['tolerant']:.1%}  failed {summary['failed']:.1%}  (n={len(recs)})")
    return summary

## 6. Train

Re-running this cell resumes from the last checkpoint in `OUT` (useful with `USE_DRIVE`).

In [ ]:
import glob, math
from transformers import Trainer, TrainingArguments


def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, value: seq + [value] * (n - len(seq))
    return {
        "input_ids": torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
        "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch]),
        "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
    }


CKPT_DIR = f"{OUT}/checkpoints"
steps_per_epoch = math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM))
total_steps = 8 if SMOKE else steps_per_epoch * EPOCHS
save_every = 4 if SMOKE else math.ceil(steps_per_epoch / CHECKPOINTS_PER_EPOCH)
args = TrainingArguments(
    output_dir=CKPT_DIR,
    max_steps=total_steps,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4 if SMOKE else GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, round(0.03 * total_steps)),
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    bf16=True,
    logging_steps=1 if SMOKE else 10,
    eval_strategy="steps",
    eval_steps=save_every,
    save_strategy="steps",
    save_steps=save_every,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=collate)
print(f"{total_steps} optimizer steps, checkpoint every {save_every}")
resume = bool(glob.glob(f"{CKPT_DIR}/checkpoint-*"))
trainer.train(resume_from_checkpoint=resume or None)
final_dir = f"{CKPT_DIR}/checkpoint-{trainer.state.global_step}"
if not os.path.exists(final_dir):
    trainer.save_model(final_dir)

## 7. Pick the checkpoint on dev

Scores the untuned base model and every checkpoint on the same dev questions. The best by within-1% accuracy (ties: scale, then the later step) is exported.

In [ ]:
import shutil

results = {"base": None, "checkpoints": {}}
with model.disable_adapter():
    results["base"] = dev_eval(model, dev_gen, "base")

ckpts = sorted(glob.glob(f"{CKPT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
for path in ckpts:
    name = "step" + path.rsplit("-", 1)[1]
    if name not in model.peft_config:
        model.load_adapter(path, adapter_name=name)
    model.set_adapter(name)
    results["checkpoints"][name] = dev_eval(model, dev_gen, name)

best = max(results["checkpoints"], key=lambda k: (results["checkpoints"][k]["tolerant"],
                                                  results["checkpoints"][k]["scale"],
                                                  int(k[4:])))
best_path = next(p for p in ckpts if p.endswith("-" + best[4:]))
BEST_DIR = f"{OUT}/best_adapter"
os.makedirs(BEST_DIR, exist_ok=True)
for fname in ("adapter_model.safetensors", "adapter_config.json"):
    shutil.copy(f"{best_path}/{fname}", BEST_DIR)
print(f"best: {best} ({results['checkpoints'][best]['tolerant']:.1%} within 1% vs "
      f"base {results['base']['tolerant']:.1%}) -> {BEST_DIR}")

In [ ]:
import bitsandbytes, peft, transformers

run_info = {
    "base_model": BASE_MODEL, "smoke": SMOKE, "repo_commit": REPO_COMMIT, "branch": BRANCH,
    "prompt_sha256": PROMPT_SHA, "prompt_suffix": " /no_think", "enable_thinking": False,
    "hyperparameters": {"epochs": EPOCHS, "lr": LR, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
                        "lora_dropout": LORA_DROPOUT, "batch": BATCH, "grad_accum": args.gradient_accumulation_steps,
                        "max_steps": total_steps, "max_grad_norm": 0.3, "scheduler": "cosine",
                        "optim": "paged_adamw_8bit", "quant": "nf4 double-quant, bf16 compute",
                        "target_modules": "q,k,v,o,gate,up,down", "seed": SEED},
    "n_train": len(train_ds), "dev_gen_ids": [e.id for e in dev_gen],
    "dev_results": results, "best_checkpoint": best,
    "log_history": trainer.state.log_history,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "peft": peft.__version__, "bitsandbytes": bitsandbytes.__version__,
                 "gpu": torch.cuda.get_device_name(0)},
}
with open(f"{OUT}/run_info.json", "w") as fh:
    json.dump(run_info, fh, indent=2, default=str)
print("wrote", f"{OUT}/run_info.json")

## 8. Export: merge, convert, quantize

Merges the best adapter into the bf16 base model, converts it with llama.cpp and quantizes
to Q4_K_M. The training model is freed first; the merge runs on the GPU (16 GB of bf16
weights). Needs about 40 GB of free disk. Takes about 15 minutes on an A100.

Note: dev scores above are for the 4-bit NF4 base plus adapter; the laptop runs the merged
Q4_K_M model, so its test score also reflects that quantization (as the baseline's did).

In [ ]:
import gc, shutil
from peft import PeftModel

LLAMA_CPP = "/content/llama.cpp"
if not os.path.exists(LLAMA_CPP):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA_CPP], check=True)
%pip install -q {LLAMA_CPP}/gguf-py
LLAMA_CPP_COMMIT = subprocess.run(["git", "-C", LLAMA_CPP, "rev-parse", "HEAD"],
                                  capture_output=True, text=True).stdout.strip()

del trainer, model
gc.collect(); torch.cuda.empty_cache()
MERGED = f"{OUT}/merged"
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map={"": 0})
merged = PeftModel.from_pretrained(base, BEST_DIR).merge_and_unload()
merged.save_pretrained(MERGED)
tok.save_pretrained(MERGED)
del base, merged
gc.collect(); torch.cuda.empty_cache()

BF16_GGUF = f"{OUT}/merged-bf16.gguf"
subprocess.run([sys.executable, f"{LLAMA_CPP}/convert_hf_to_gguf.py", MERGED,
                "--outtype", "bf16", "--outfile", BF16_GGUF], check=True)
subprocess.run(f"cmake -S {LLAMA_CPP} -B {LLAMA_CPP}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF "
               f"&& cmake --build {LLAMA_CPP}/build --target llama-quantize -j", shell=True, check=True)
subprocess.run([f"{LLAMA_CPP}/build/bin/llama-quantize", BF16_GGUF, f"{OUT}/{GGUF_NAME}", "Q4_K_M"],
               check=True)
shutil.rmtree(MERGED)
os.remove(BF16_GGUF)

run_info["llama_cpp_commit"] = LLAMA_CPP_COMMIT
run_info["gguf"] = {"file": GGUF_NAME, "quant": "Q4_K_M", "merged_from": best}
with open(f"{OUT}/run_info.json", "w") as fh:
    json.dump(run_info, fh, indent=2, default=str)
print(f"{OUT}/{GGUF_NAME}: {os.path.getsize(f'{OUT}/{GGUF_NAME}') / 2**30:.1f} GB")

## 9. Get the artifacts to the laptop

Uploads to a private Hugging Face repo if `HF_REPO` is set. Otherwise zips them; a 5 GB browser download can be unreliable, so prefer Hugging Face (or `USE_DRIVE`).

In [ ]:
artifacts = [GGUF_NAME, "run_info.json", "best_adapter/adapter_config.json",
             "best_adapter/adapter_model.safetensors"]
artifacts += [os.path.basename(p) for p in glob.glob(f"{OUT}/dev_*.jsonl")]

if HF_REPO:
    from getpass import getpass
    from huggingface_hub import HfApi

    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = getpass("Hugging Face write token: ")
    api = HfApi(token=token)
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO, folder_path=OUT, allow_patterns=artifacts,
                      commit_message=f"FinQA QLoRA {'smoke ' if SMOKE else ''}{best}")
    print(f"uploaded to https://huggingface.co/{HF_REPO}")
else:
    import zipfile

    bundle = "/content/finqa_qlora_artifacts.zip"
    with zipfile.ZipFile(bundle, "w") as zf:
        for a in artifacts:
            zf.write(f"{OUT}/{a}", a)
    print(f"{bundle}: {os.path.getsize(bundle) / 2**30:.1f} GB")
    try:
        from google.colab import files
        files.download(bundle)
    except Exception as exc:
        print("download it from the Colab file browser:", exc)

## 10. On the laptop

From the repo root in Git Bash. The Modelfile is `qwen3:8b`'s own, with only `FROM`
swapped, so the chat template, the thinking switch and the stop tokens are unchanged
(checked on the laptop: a model imported this way renders the same prompts as `qwen3:8b`).

```bash
RUN=data/finqa/models/qwen3-8b-finqa
OLLAMA="$LOCALAPPDATA/Programs/Ollama/ollama.exe"
.venv/Scripts/hf download <HF_REPO> --local-dir $RUN          # or unzip the bundle there
"$OLLAMA" show qwen3:8b --modelfile | sed "s|^FROM .*|FROM ./qwen3-8b-finqa-Q4_K_M.gguf|" > $RUN/Modelfile
(cd $RUN && "$OLLAMA" create qwen3-finqa -f Modelfile)
PYTHONUTF8=1 .venv/Scripts/python.exe scripts/finqa_baseline.py --model qwen3-finqa --modes program
```

The fine-tuned model is scored with the same prompt, options and executor as the baseline,
and lands in the same report.